# Lab 2: Building a Simple Context Provider

In this lab, you will build a context provider that extracts user information from conversations and uses it to personalize the agent's behavior. Context providers are framework-managed components with two lifecycle hooks — `before_run()` injects context before the LLM is called, and `after_run()` extracts or stores information after it responds. They run automatically on every turn without the agent needing to request them.

## What you will learn

- How to create a custom context provider by extending `BaseContextProvider`
- How to use `before_run()` to inject dynamic instructions
- How to use `after_run()` to extract structured data from conversations
- How state persists across turns in a session

## Setup

Load the environment variables from your `.env` file.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

Import the required modules from the Microsoft Agent Framework and Pydantic.

In [ ]:
from typing import Any

from agent_framework import (
    AgentSession,
    BaseContextProvider,
    SessionContext,
    SupportsChatGetResponse,
)
from llm_provider import get_client
from pydantic import BaseModel

## Define a Data Model

Create a Pydantic model for the data the provider will extract. Structured output lets the provider extract specific fields from free-form conversation, turning unstructured text into data it can act on in later turns.

In [ ]:
class UserInfo(BaseModel):
    name: str | None = None
    age: int | None = None

## Why Context Providers?

Context providers address three fundamental LLM limitations: **hallucination** (generating plausible but incorrect answers), **knowledge cutoff** (no access to your specific data), and **relationship blindness** (inability to traverse connections between entities). Unlike tools, which the agent must choose to call, context providers run automatically on every turn — ensuring relevant knowledge is always available.

## Create the Context Provider

Build a `UserInfoMemory` class that extends `BaseContextProvider`. It should:

1. Accept a chat client in `__init__` and call `super().__init__("user-info-memory")`
2. Implement `before_run()` to check state for known user info and inject instructions:
   - The `state` dict is already scoped to this provider (keyed by `source_id`), so store values directly in it
   - If the name is unknown, tell the agent to ask for it
   - If the name is known, tell the agent the user's name
   - Same for age
   - Use `context.extend_instructions()` to inject the instructions
3. Implement `after_run()` to extract user info from the conversation:
   - Skip extraction if both name and age are already known
   - Use `context.get_messages()` to get conversation messages
   - Use the chat client's `get_response()` with `response_format: UserInfo` to extract structured data
   - Update state with any newly extracted values

In [ ]:
class UserInfoMemory(BaseContextProvider):
    """Context provider that extracts and remembers user info (name, age)."""

    def __init__(self, client: SupportsChatGetResponse):
        super().__init__("user-info-memory")
        self._chat_client = client

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Inject dynamic instructions based on stored user info."""
        user_info = state.setdefault("user_info", UserInfo())

        instructions: list[str] = []

        if user_info.name is None:
            instructions.append(
                "Ask the user for their name and politely decline "
                "to answer any questions until they provide it."
            )
        else:
            instructions.append(f"The user's name is {user_info.name}.")

        if user_info.age is None:
            instructions.append(
                "Ask the user for their age and politely decline "
                "to answer any questions until they provide it."
            )
        else:
            instructions.append(f"The user's age is {user_info.age}.")

        context.extend_instructions(self.source_id, " ".join(instructions))

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Extract user info from the conversation after each turn."""
        user_info = state.setdefault("user_info", UserInfo())
        if user_info.name is not None and user_info.age is not None:
            return

        request_messages = context.get_messages(
            include_input=True, include_response=True
        )
        user_messages = [
            msg for msg in request_messages
            if hasattr(msg, "role") and msg.role == "user"
        ]
        if not user_messages:
            return

        try:
            result = await self._chat_client.get_response(
                messages=request_messages,
                instructions=(
                    "Extract the user's name and age from the message "
                    "if present. If not present return nulls."
                ),
                options={"response_format": UserInfo},
            )
            extracted = result.value
            if extracted and user_info.name is None and extracted.name:
                user_info.name = extracted.name
            if extracted and user_info.age is None and extracted.age:
                user_info.age = extracted.age
            state["user_info"] = user_info
        except Exception:
            pass

## Create and Run the Agent

Create an agent with the `UserInfoMemory` context provider and run a multi-turn conversation.

The conversation tests three turns:
1. The user asks a question without providing their name — the provider should make the agent ask for it
2. The user provides their name and age
3. The user asks the same question again — the agent should now answer and address them by name

In [ ]:
client = get_client()

agent = client.as_agent(
    name="context-provider-agent",
    instructions=(
        "You are a friendly assistant. Always address the user "
        "by their name when you know it."
    ),
    context_providers=[UserInfoMemory(client)],
)

session = agent.create_session()

queries = [
    "Hello, what is the square root of 9?",
    "My name is Alex and I am 30 years old",
    "Now, what is the square root of 9?",
]

for query in queries:
    print(f"User: {query}\n")
    print("Answer: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

## Inspect the Extracted State

After the conversation, check that the context provider successfully extracted the user's name and age from the session state.

In [ ]:
user_info = session.state.get("user-info-memory", {}).get(
    "user_info", UserInfo()
)
print(f"Extracted Name: {user_info.name}")
print(f"Extracted Age: {user_info.age}")

## Summary

You built a `UserInfoMemory` context provider that:

- Uses `before_run()` to inject dynamic instructions based on what it knows about the user
- Uses `after_run()` to extract structured data (name and age) from conversations
- Persists state across turns using the provider-scoped `state` dictionary (the framework automatically namespaces it under `session.state[source_id]`)

Unlike tools, context providers run **automatically** on every turn — the agent does not need to decide to call them.

**What's next:** In the next module, you will use `Neo4jContextProvider` to inject knowledge graph context using vector, fulltext, and hybrid search.